In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
# Load main dataset
df = pd.read_csv("merged_output.csv")

# Convert date
df["CreatedOn"] = pd.to_datetime(df["CreatedOn"], errors="coerce")

print("Shape:", df.shape)
print("Date range:", df["CreatedOn"].min(), "to", df["CreatedOn"].max())

Shape: (460899, 6)
Date range: 2013-01-01 00:00:00 to 2017-09-26 00:00:00


In [3]:
aphid_df = df[df["Pest"].str.lower() == "aphid"].copy()

print("Aphid shape:", aphid_df.shape)
print("Unique districts:", aphid_df["district"].nunique())

Aphid shape: (42500, 6)
Unique districts: 509


In [4]:
aphid_daily = (
    aphid_df
    .groupby(["CreatedOn", "district"])["pest_count"]
    .sum()
    .reset_index()
)

pivot_df = aphid_daily.pivot(
    index="CreatedOn",
    columns="district",
    values="pest_count"
)

pivot_df = pivot_df.sort_index().fillna(0)

print("Pivot shape:", pivot_df.shape)

Pivot shape: (1701, 509)


In [5]:
pivot_rolling = pivot_df.rolling(window=7, min_periods=1).sum()

print("Rolling shape:", pivot_rolling.shape)

Rolling shape: (1701, 509)


In [6]:
latlon = pd.read_csv("latlon_df.csv")

common_districts = list(
    set(pivot_rolling.columns).intersection(set(latlon["district"]))
)

pivot_filtered = pivot_rolling[common_districts]

latlon_aligned = latlon[latlon["district"].isin(common_districts)].copy()
latlon_aligned = latlon_aligned.set_index("district")
latlon_aligned = latlon_aligned.loc[pivot_filtered.columns]

print("Final district count:", pivot_filtered.shape[1])

Final district count: 486


In [7]:
outbreak_df = pd.DataFrame(index=pivot_filtered.index)

for col in pivot_filtered.columns:
    threshold = pivot_filtered[col].quantile(0.75)
    outbreak_df[col] = (pivot_filtered[col] > threshold).astype(int)

print("Overall outbreak percentage:",
      outbreak_df.values.mean() * 100)

C:\Users\sajal\AppData\Local\Temp\ipykernel_28960\525283528.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outbreak_df[col] = (pivot_filtered[col] > threshold).astype(int)
C:\Users\sajal\AppData\Local\Temp\ipykernel_28960\525283528.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outbreak_df[col] = (pivot_filtered[col] > threshold).astype(int)
C:\Users\sajal\AppData\Local\Temp\ipykernel_28960\525283528.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many tim

Overall outbreak percentage: 11.975405413905643


C:\Users\sajal\AppData\Local\Temp\ipykernel_28960\525283528.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outbreak_df[col] = (pivot_filtered[col] > threshold).astype(int)
C:\Users\sajal\AppData\Local\Temp\ipykernel_28960\525283528.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outbreak_df[col] = (pivot_filtered[col] > threshold).astype(int)
C:\Users\sajal\AppData\Local\Temp\ipykernel_28960\525283528.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many tim

In [8]:
scalers = {}
scaled_data = np.zeros_like(pivot_filtered.values)

for i, col in enumerate(pivot_filtered.columns):
    scaler = MinMaxScaler()
    scaled_col = scaler.fit_transform(
        pivot_filtered[col].values.reshape(-1,1)
    )
    scaled_data[:, i] = scaled_col.flatten()
    scalers[col] = scaler

In [9]:
window_size = 30

X = []
y = []

for i in range(len(scaled_data) - window_size):
    X.append(scaled_data[i:i+window_size])
    y.append(outbreak_df.values[i+window_size])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1671, 30, 486)
y shape: (1671, 486)


In [10]:
split_index = int(0.8 * len(X))

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (1336, 30, 486)
Test: (335, 30, 486)


In [11]:
num_nodes = X.shape[2]

inputs = layers.Input(shape=(30, num_nodes))

x = layers.LSTM(64, return_sequences=False)(inputs)

outputs = layers.Dense(num_nodes, activation='sigmoid')(x)

baseline_model = models.Model(inputs, outputs)

baseline_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

baseline_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 30, 486)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │       141,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 486)            │        31,590 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 172,646 (674.40 KB)

 Trainable params: 172,646 (674.40 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 7.4850e-04 - loss: 0.5028 - val_accuracy: 0.0000e+00 - val_loss: 0.3489
Epoch 2/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0000e+00 - loss: 0.3500 - val_accuracy: 0.0000e+00 - val_loss: 0.3392
Epoch 3/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0000e+00 - loss: 0.3455 - val_accuracy: 0.0000e+00 - val_loss: 0.3402
Epoch 4/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0000e+00 - loss: 0.3438 - val_accuracy: 0.0000e+00 - val_loss: 0.3367
Epoch 5/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0000e+00 - loss: 0.3405 - val_accuracy: 0.0000e+00 - val_loss: 0.3353
Epoch 6/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.0000e+00 - loss: 0.3348 - val_accuracy: 0.0000e+00 - val_loss: 0.3294
Epoch 7/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.0000e+00 - loss: 0.3294 - val_accuracy: 0.0000e+00 - val_loss: 0.3266
Epoch 8/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - ac

In [13]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

y_pred_prob = baseline_model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print("Accuracy:",
      accuracy_score(y_test.flatten(), y_pred.flatten()))

print("F1 Score:",
      f1_score(y_test.flatten(), y_pred.flatten()))

print("ROC-AUC:",
      roc_auc_score(y_test.flatten(), y_pred_prob.flatten()))

11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 52ms/step
Accuracy: 0.8936060438548
F1 Score: 0.2093299251415008
ROC-AUC: 0.7781025672393141


In [14]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

precision = precision_score(y_test.flatten(), y_pred.flatten())
recall = recall_score(y_test.flatten(), y_pred.flatten())

cm = confusion_matrix(y_test.flatten(), y_pred.flatten())

print("Precision:", precision)
print("Recall:", recall)
print("Confusion Matrix:\n", cm)

Precision: 0.6019952743502232
Recall: 0.12669208243549368
Confusion Matrix:
 [[143195   1516]
 [ 15806   2293]]


In [15]:
class_weight = {0:1, 1:5}

history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    class_weight=class_weight,
    verbose=1
)

Epoch 1/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.0374 - loss: 0.2632 - val_accuracy: 0.0000e+00 - val_loss: 0.2947
Epoch 2/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0262 - loss: 0.2599 - val_accuracy: 0.0000e+00 - val_loss: 0.2944
Epoch 3/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0269 - loss: 0.2569 - val_accuracy: 0.0000e+00 - val_loss: 0.2939
Epoch 4/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.0262 - loss: 0.2539 - val_accuracy: 0.0000e+00 - val_loss: 0.2937
Epoch 5/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.0232 - loss: 0.2511 - val_accuracy: 0.0000e+00 - val_loss: 0.2938
Epoch 6/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.0180 - loss: 0.2484 - val_accuracy: 0.0000e+00 - val_loss: 0.2933
Epoch 7/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.0240 - loss: 0.2458 - val_accuracy: 0.0000e+00 - val_loss: 0.2937
Epoch 8/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.0135 - loss: 0.243

In [16]:
y_pred = (y_pred_prob > 0.3)

In [17]:
threshold = 0.3
y_pred_adj = (y_pred_prob > threshold).astype(int)

print("Precision:",
      precision_score(y_test.flatten(), y_pred_adj.flatten()))

print("Recall:",
      recall_score(y_test.flatten(), y_pred_adj.flatten()))

print("F1:",
      f1_score(y_test.flatten(), y_pred_adj.flatten()))

Precision: 0.41579249892225895
Recall: 0.3197414221780209
F1: 0.3614954555392448


In [18]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.9, 0.05)
best_f1 = 0
best_threshold = 0

for t in thresholds:
    y_pred_temp = (y_pred_prob > t).astype(int)
    f1 = f1_score(y_test.flatten(), y_pred_temp.flatten())
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

Best threshold: 0.20000000000000004
Best F1: 0.38082486446146757


In [22]:
num_nodes = X.shape[2]

A_tensor = tf.constant(A_norm.astype(np.float32))

inputs = layers.Input(shape=(30, num_nodes))

def spatial_layer(x):
    return tf.einsum('ij,bwj->bwi', A_tensor, x)

x = layers.Lambda(spatial_layer)(inputs)

x = layers.LSTM(64, return_sequences=False)(x)

outputs = layers.Dense(num_nodes, activation='sigmoid')(x)

spatial_model = models.Model(inputs, outputs)

spatial_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

spatial_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 30, 486)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 30, 486)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │       141,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 486)            │        31,590 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 172,646 (674.40 KB)

 Trainable params: 172,646 (674.40 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
# Extract coordinates in correct order
coords = latlon_aligned[["latitude", "longitude"]].values

# Convert to radians for haversine
coords_rad = np.radians(coords)

from sklearn.neighbors import NearestNeighbors

k = 5

nbrs = NearestNeighbors(
    n_neighbors=k+1,
    algorithm='ball_tree',
    metric='haversine'
)

nbrs.fit(coords_rad)
distances, indices = nbrs.kneighbors(coords_rad)

num_nodes = coords.shape[0]
A = np.zeros((num_nodes, num_nodes))

for i in range(num_nodes):
    for j in indices[i][1:]:  # skip itself
        A[i, j] = 1
        A[j, i] = 1

# Add self-loops
A = A + np.eye(num_nodes)

print("Adjacency shape:", A.shape)

Adjacency shape: (486, 486)


In [21]:
# Degree matrix
D = np.diag(A.sum(axis=1))

D_inv_sqrt = np.linalg.inv(np.sqrt(D))

A_norm = D_inv_sqrt @ A @ D_inv_sqrt

print("Normalized adjacency shape:", A_norm.shape)

Normalized adjacency shape: (486, 486)


In [23]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_spatial = spatial_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.0000e+00 - loss: 0.4980 - val_accuracy: 0.0000e+00 - val_loss: 0.3471
Epoch 2/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.0000e+00 - loss: 0.3501 - val_accuracy: 0.0000e+00 - val_loss: 0.3414
Epoch 3/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.0000e+00 - loss: 0.3456 - val_accuracy: 0.0000e+00 - val_loss: 0.3399
Epoch 4/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.0000e+00 - loss: 0.3447 - val_accuracy: 0.0000e+00 - val_loss: 0.3393
Epoch 5/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.0000e+00 - loss: 0.3438 - val_accuracy: 0.0000e+00 - val_loss: 0.3406
Epoch 6/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.0000e+00 - loss: 0.3426 - val_accuracy: 0.0000e+00 - val_loss: 0.3369
Epoch 7/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.0000e+00 - loss: 0.3412 - val_accuracy: 0.0000e+00 - val_loss: 0.3348
Epoch 8/30
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - ac

In [24]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

y_pred_prob_sp = spatial_model.predict(X_test)

# ROC-AUC
roc_sp = roc_auc_score(y_test.flatten(), y_pred_prob_sp.flatten())
print("Spatial ROC-AUC:", roc_sp)

# Find best threshold for F1
thresholds = np.arange(0.1, 0.9, 0.05)
best_f1 = 0
best_threshold = 0

for t in thresholds:
    y_pred_temp = (y_pred_prob_sp > t).astype(int)
    f1 = f1_score(y_test.flatten(), y_pred_temp.flatten())
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Spatial Best Threshold:", best_threshold)
print("Spatial Best F1:", best_f1)

11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step
Spatial ROC-AUC: 0.764383823047207
Spatial Best Threshold: 0.20000000000000004
Spatial Best F1: 0.35784033382050795
